In [ ]:
import sys
from pathlib import Path

# root first so `horizon` and `eval` resolve as packages (eval/ imports `from horizon...`);
# horizon/ also on sys.path so the package's internal `from fds...`/`import utils...` resolve.
# root MUST precede horizon/ — else `import horizon` binds to horizon/horizon.py, not the
# package ("horizon is not a package").
root = Path("..").resolve()
sys.path.insert(0, str(root / "horizon"))
sys.path.insert(0, str(root))

from horizon.utils.loaders import get_fds
from eval.report import characterize_lazy

In [ ]:
# pick a dataset — edit these two paths
fds_path = Path("..") / "datasets" / "complaints_10m" / "fds.csv"
data_path = Path("..") / "datasets" / "complaints_10m" / "clean.parquet"

In [ ]:
import polars as pl

scan = pl.scan_parquet if data_path.suffix.lower() == ".parquet" else pl.scan_csv
print(scan(data_path).collect_schema().names())

In [ ]:
fds = get_fds(fds_path, data_path)
print(f"Loaded {len(fds)} FDs")
fds

In [ ]:
# characterize_lazy reads only attr(fd) per FD and prints progress.
result = characterize_lazy(data_path, fds)

In [ ]:
# drop FDs whose LHS redundancy is below MIN_REDUNDANCY — too few repeats to
# support a pattern (avg LHS group size ≈ n^MIN_REDUNDANCY). result already
# contains fd_lhs_redundancy, so this is a pure-Python filter on the dict.
import pandas as pd

MIN_REDUNDANCY = 0.0

red = result["fd_lhs_redundancy"]
kept = [fd for fd in fds if red[fd.lhs_attributes] >= MIN_REDUNDANCY]
dropped = [fd for fd in fds if red[fd.lhs_attributes] < MIN_REDUNDANCY]
print(f"kept {len(kept)} / {len(fds)} FDs; dropping {len(dropped)} (red < {MIN_REDUNDANCY}):")
for fd in dropped:
    print(f"  {red[fd.lhs_attributes]:.3f}  {fd}")

result["g3_error"] = {fd: v for fd, v in result["g3_error"].items() if fd in kept}
result["violation_clusters"] = {fd: v for fd, v in result["violation_clusters"].items() if fd in kept}
result["fd_lhs_redundancy"] = {lhs: v for lhs, v in red.items() if any(fd.lhs_attributes == lhs for fd in kept)}

pd.DataFrame(
    [{"from": ";".join(fd.lhs_attributes), "to": fd.rhs} for fd in kept]
).to_csv(fds_path, index=False)
fds = kept

In [ ]:
from pprint import pprint

summary = {k: v for k, v in result.items() if k not in {"violation_clusters"}}
pprint(summary, sort_dicts=False)

In [ ]:
# inspect violations for one FD: pick by index
i = 13
fd = fds[i]
clusters = result["violation_clusters"][fd]
print(f"[{i}] {fd}   G3 = {result['g3_error'][fd]:.4f}   {clusters.height} violation rows")
pl.Config.set_fmt_str_lengths(200)  # don't truncate string cells in the display
pl.Config.set_tbl_rows(300)  # show all 30 rows, not Polars' default 10
clusters.tail(300)